# Lecture 12. Hands-on with e3nn-jax

**PHYG004 · Sogang University · 2026 Spring**
**Instructor:** Prof. Young Woo Choi

**Data type:** *toy/synthetic* (Tetris benchmark) **+ real data** (rMD17 aspirin trajectory, energies in eV, forces in eV/Å).

This notebook is the working chapter for Week 12. It is meant to be read on its own as a textbook section, with code cells used as worked examples. The narrative is self-contained; the roadmap and notation summary below fix the symbols used throughout.

## 12.1 Learning goals

After working through this chapter you should be able to:

1. State what *equivariance under rotation* means for a function defined on point clouds, and recognise it as a structural constraint, not a learning objective.
2. Name the irreducible representations of $\mathrm{O}(3)$ as labelled by $\ell \in \mathbb{Z}_{\geq 0}$ and parity, and know the dimensions $2\ell+1$.
3. Use spherical harmonics $Y_\ell(\hat r)$ as the elementary equivariant feature on the unit sphere, and use the Wigner-D matrices as the corresponding rotation rule.
4. Read an `e3nn` *irreps string* such as `32x0e + 8x1o + 8x2e` as a typed-feature declaration.
5. Combine two equivariant features with the *tensor product* and read off the Clebsch–Gordan output decomposition.
6. Replace SchNet's distance-only message by a direction-aware tensorial message and explain why this restores rotation behaviour and chirality discrimination simultaneously.
7. Run a small Tensor-Field-Network classifier on the Tetris benchmark and confirm exact equivariance to floating-point precision.
8. Train the **same** TFN on a *real* molecular trajectory (rMD17 aspirin) to regress the potential energy in physical units, and compare the toy and real performance side by side.

## 12.2 Where this fits

L11 built two non-equivariant baselines on the eight-shape Tetris point-cloud benchmark and uncovered two architectural failure modes:

| Model (L11) | train acc | rotated-test acc | chiral pair |
|---|---|---|---|
| flat-coordinate MLP | 100% | $\approx$ 12% (random) | overfits, useless on rotation |
| SchNet (distance-only MPNN) | 87.5% | 87.5% | logits identical — invisible |

The diagnosis from L11.

* Flat coordinates are *rotation-sensitive but unstructured*: a generic MLP must learn rotation behaviour from data, which 8 examples cannot supply.
* The pairwise-distance multiset $\{r_{ij}\}$ that SchNet uses is *invariant under reflection*, so chiral mirror images are encoded identically.

Both failure modes have the same cure. Replace the SchNet message
$$m_{ij}^{\text{SchNet}} \;=\; \big(W^{(s)} h_j\big) \odot \mathrm{MLP}\!\big(\mathrm{RBF}(r_{ij})\big)$$
by the *equivariant message*
$$\boxed{\;m_{ij}^{\text{equiv}} \;=\; h_j \,\otimes\, Y_\ell(\hat r_{ij})\;}$$
which carries the bond *direction* through the spherical harmonics. The remainder of the chapter is the math machinery that gives this expression a precise meaning, and a working implementation that demonstrates the cure.

![rotated molecule. Equivariant network produces correctly rotated forces. Vanilla MLP outputs random forces.](images/00_motivation.png)

## 12.3 Why equivariance, and not data augmentation

Two routes can teach a network to behave correctly under rotation.

* **Augmentation.** Train on many rotated copies of every input. Simple to implement; statistically inefficient; never gives an exact guarantee on rotations not seen at training time.
* **Built-in equivariance.** Every layer is designed so that rotating the input rotates the output in the appropriate way. Correct on every rotation by construction, with no augmentation needed.

The second route is the design choice taken by NequIP (Batzner et al., 2022), MACE (Batatia et al., 2022) and the foundation model MACE-MP-0 (2023). On molecular benchmarks these models match or beat non-equivariant baselines using two-to-three orders of magnitude less training data. The reason is mathematical, not empirical: a rotation-equivariant model has a smaller hypothesis class — it cannot represent rotation-violating functions — so the same number of training points covers it more thoroughly.

## 12.4 Notation

The same symbols recur throughout.

| Symbol | Meaning |
|---|---|
| $R \in \mathrm{SO}(3)$ | a $3\times 3$ rotation matrix; $R^\top R = I$, $\det R = +1$ |
| $\hat r$ | a unit vector in $\mathbb{R}^3$ |
| $\ell \in \{0, 1, 2, \ldots\}$ | irreducible-representation index (angular-momentum quantum number) |
| $Y_\ell^m(\hat r)$ | real spherical harmonic, $m = -\ell, \ldots, +\ell$ |
| $Y_\ell(\hat r)$ | $(2\ell+1)$-vector stacking $Y_\ell^m$ over $m$ |
| $D^\ell(R)$ | Wigner-D matrix; $(2\ell+1)\times(2\ell+1)$ |
| $h_i$ | feature carried by atom $i$, an `IrrepsArray` |
| $\vec r_{ij},\, r_{ij},\, \hat r_{ij}$ | displacement *from sender $j$ to receiver $i$*, $\vec r_{ij} := \vec r_i - \vec r_j$; its length and unit direction |
| $\otimes,\,\oplus$ | equivariant tensor product, direct sum of irreps |
| `0e`, `0o`, `1e`, `1o`, … | parity-even/odd irreps; e/o = even/odd under inversion |

## 12.5 Source material

* Reference code. [`e3nn-jax/examples/tetris_point.py`](https://github.com/e3nn/e3nn-jax/blob/main/examples/tetris_point.py)
* Library docs. <https://e3nn-jax.readthedocs.io/>
* TFN paper. Thomas et al., 2018. *Tensor Field Networks*. <https://arxiv.org/abs/1802.08219>

## 12.6 Runtime

Use **Colab CPU**. The Tetris benchmark has eight examples; GPU setup adds latency without speedup. The first JAX-compiled step takes 30–90 s; subsequent steps are subsecond.


## 12.0 Recap: one symmetry blueprint, three architectures

In L11 we drew a single *geometric deep learning* blueprint: **bake the symmetry of the
data into the architecture, so the model cannot represent symmetry-violating functions.**
Each family we have met is one instance of that blueprint, differing only in *which group*
it respects.

| Lecture | Architecture | Symmetry group | What is enforced |
|---|---|---|---|
| L06 | CNN | translations $\mathbb{T}(2)$ | **equivariance** — shift the image, the feature map shifts the same way |
| L11 | GNN / SchNet | permutations $S_n$ | **invariance** — relabel the atoms, the energy is unchanged |
| **L12 (today)** | **e3nn / TFN** | **Euclidean group $\mathrm{E}(3)=\mathrm{O}(3)\ltimes\mathbb{R}^3$** | **full equivariance** — rotate / reflect / translate the molecule, scalars stay fixed and vectors (forces) rotate with it |

Today's case is the richest: $\mathrm{E}(3)$ contains **rotations** (the $\mathrm{SO}(3)$
piece) *and* **parity / reflection** (the extra $\mathbb{Z}_2$ that makes it $\mathrm{O}(3)$).
Rotation equivariance is what fixes the data-hungry flat-MLP baseline of L11; parity is what
lets the model tell a left-handed molecule from its mirror image — a distinction that
distance-only SchNet is *blind* to.

So read this lecture as the **rotation + parity** entry in the blueprint table. Everything
else (message passing, permutation invariance over neighbours, the three-step MPNN template)
is inherited unchanged from L11.

> **Today's payoff.** After the Tetris warm-up that proves the machinery works to machine
> precision, we point the *same* network at a **real molecule** — the rMD17 aspirin
> trajectory — and regress its DFT potential energy in eV. That is the first time in this
> course an equivariant network touches genuine atomistic coordinates with physical units.

## 12.7 Cell 1 — Install packages

Four small packages cover everything in this chapter.

* `e3nn-jax`. Equivariant-neural-network library. The main object of study.
* `flax`. A thin neural-network wrapper around JAX (Linen API).
* `optax`. JAX optimizers. We use Adam.
* `jraph`. JAX graph utilities. Provides the `GraphsTuple` data structure and the `GraphNetwork` template used in §12.16.

`jax` itself is **not** reinstalled. Colab ships with a `jax`/`jaxlib` pair matched to its accelerators; replacing them by `pip install jax` can break GPU support.


In [ ]:
# Core equivariant stack. (jax/jaxlib are pre-installed on Colab — do not reinstall.)
!pip install -q "e3nn-jax>=0.20.8,<0.22" flax optax jraph tqdm matplotlib

# rMD17 (§12.A) only needs numpy + a small download — no extra package required.
# Optional alternative loaders for rMD17 if the figshare URL is blocked:
#   !pip install -q mace-torch ase


## 12.8 Cell 2 — Imports and setup

Two non-default settings are worth justifying.

1. **`float64` mode.** Several checks below verify that two computations agree to about $10^{-15}$, which is float64 machine precision. Float32 ($\approx 10^{-7}$) cannot resolve the equivariance identities at that scale, and a small discrepancy could be confused with a bug.
2. **Fixed PRNG seed.** All random samples (rotation matrices, weight initialisations) are derived from `jax.random.PRNGKey(42)` so that the numbers printed in the chapter match the numbers a student running the cell will see.


In [ ]:
import jax
jax.config.update("jax_enable_x64", True)

import jax.numpy as jnp
import numpy as np
import e3nn_jax as e3nn
import flax.linen as nn
import jraph
import optax
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

SEED = 42
key = jax.random.PRNGKey(SEED)

print("jax version  :", jax.__version__)
print("e3nn-jax     :", e3nn.__version__)
print("devices      :", jax.devices())
print("default dtype:", jnp.zeros(1).dtype)

## 12.9 Cell 3 — Rotations as orthogonal matrices *(quick aside)*

> **Aside (no new ideas).** A rotation $R\in\mathrm{SO}(3)$ is a $3\times3$ matrix with $R^\top R = I$ and $\det R = +1$. The cell below just draws a random $R$ and prints $\lVert R^\top R - I\rVert = O(\varepsilon)$ to confirm orthogonality numerically — skim it and move on. Equivariance arguments use only the two boxed identities below.

<details><summary>Expand: full derivation (length preservation ⇔ orthogonality)</summary>


A *rotation* in three dimensions is a linear map $R : \mathbb{R}^3 \to \mathbb{R}^3$ that

* preserves the length of every vector, $|R\vec v| = |\vec v|$, and
* preserves orientation, $\det R = +1$.

In a Cartesian basis $R$ is a $3\times 3$ matrix and the length-preservation condition is equivalent to *orthogonality*,
$$R^\top R \;=\; I,$$
because
$$|R\vec v|^2 \;=\; (R\vec v)^\top (R\vec v) \;=\; \vec v^{\top} R^\top R\, \vec v \;=\; |\vec v|^2 \quad \text{for all } \vec v \;\Longleftrightarrow\; R^\top R = I.$$
The set of all such $R$ forms a Lie group, the **rotation group** $\mathrm{SO}(3)$. Its dimension is three: a rotation is fixed by, e.g., an axis (two angles) and a rotation angle about that axis. We will treat $R$ as a black box that obeys the two displayed identities; equivariance arguments use only those.

> **Active vs passive.** Throughout this chapter $R$ acts on points: $\vec r \mapsto R\vec r$. This is the *active* convention — points move, the coordinate frame stays fixed. The opposite convention (frame rotates, points stay fixed) gives identical equations with $R$ replaced by $R^\top$.

</details>

The cell below draws a uniformly random rotation from $\mathrm{SO}(3)$ via `e3nn.rand_matrix` and verifies length preservation and orthogonality numerically.

![a vector $\vec v$ and its rotated image $R\vec v$ on the unit sphere.](images/03_rotation_vector.png)


In [ ]:
k1, k2 = jax.random.split(key)

R: jnp.ndarray = e3nn.rand_matrix(k1)        # (3, 3) rotation matrix
v: jnp.ndarray = jax.random.normal(k2, (3,))
v = v / jnp.linalg.norm(v)                   # unit vector

Rv = R @ v

print(f"R       shape: {R.shape}")
print(f"v       shape: {v.shape}")
print(f"Rv      shape: {Rv.shape}")
print(f"|Rv|         : {jnp.linalg.norm(Rv):.6f}  (should be 1)")
print(f"|R^T R - I|  : {jnp.linalg.norm(R.T @ R - jnp.eye(3)):.2e}  (should be ~0)")

## 12.10 Cell 4 — Spherical harmonics

The spherical harmonics
$$Y_\ell^m(\hat r), \qquad \ell \in \{0,1,2,\ldots\}, \quad m \in \{-\ell, -\ell+1, \ldots, +\ell\},$$
are real-valued functions on the unit sphere $S^2 \subset \mathbb{R}^3$. They are the angular eigenfunctions of the Laplace–Beltrami operator on $S^2$, with eigenvalues $-\ell(\ell+1)$, and equivalently the angular factor of solutions to the hydrogen-atom Schrödinger equation. From a physicist's perspective they are the **most natural basis of square-integrable functions on the sphere that respects rotation**.

Three facts are used below; the proofs are standard and not repeated here.

* **Dimension.** For each $\ell$ there are $2\ell+1$ independent functions, so the index $m$ runs over $2\ell+1$ values.
* **Parity.** Under inversion $\hat r \mapsto -\hat r$,
$$Y_\ell^m(-\hat r) \;=\; (-1)^\ell\, Y_\ell^m(\hat r).$$
Even $\ell$ are parity-even (label `e`); odd $\ell$ are parity-odd (label `o`). The chiral pair in the Tetris dataset is exactly the case in which parity matters; we will return to it in §§12.14 and 12.16.
* **Familiar special cases.**

| $\ell$ | $2\ell+1$ | shape | physics name |
|---|---|---|---|
| 0 | 1 | constant | $s$ orbital |
| 1 | 3 | $\propto x, y, z$ | $p$ orbitals |
| 2 | 5 | quadratic | $d$ orbitals |
| 3 | 7 | cubic | $f$ orbitals |

The cell below evaluates the five $\ell = 2$ functions on a sphere and renders them with a sign-coloured colour map. They are the familiar $d$-orbital lobes.

The reason these functions are used as building blocks (rather than any other complete basis on $S^2$, such as Fourier on a longitude grid) is the property derived in §12.11: rotation acts on the $(2\ell+1)$-vector $Y_\ell$ by a *fixed* $(2\ell+1)\times(2\ell+1)$ matrix, never mixing different $\ell$. Each $\ell$-block is closed under rotation and cannot be split further. This is what *irreducible representation of $\mathrm{SO}(3)$* means in concrete terms.

![the spherical harmonic zoo for $\ell = 0, 1, 2, 3$.](images/04_sh_zoo.png)


In [ ]:
# Sample a grid of points on the sphere
n_theta, n_phi = 60, 120
theta = jnp.linspace(0.0, jnp.pi, n_theta)
phi = jnp.linspace(0.0, 2 * jnp.pi, n_phi)
th, ph = jnp.meshgrid(theta, phi, indexing="ij")
x = jnp.sin(th) * jnp.cos(ph)
y = jnp.sin(th) * jnp.sin(ph)
z = jnp.cos(th)
rhat = jnp.stack([x, y, z], axis=-1)         # (n_theta, n_phi, 3)
rhat = e3nn.IrrepsArray("1o", rhat)         # tag these as 3-vectors (1o)
print(f"rhat shape: {rhat.array.shape}")

# spherical_harmonics returns an IrrepsArray with irreps "1o + 2e + 3o"
sh = e3nn.spherical_harmonics([1, 2, 3], rhat, normalize=True)
print(f"sh irreps : {sh.irreps}  (lmax={sh.irreps.lmax})")
print(f"sh shape  : {sh.array.shape}  (last dim = 3 + 5 + 7 = 15)")

# The l=2 block is at offset 3..8 in the array (after the l=1 block of size 3)
y2 = np.asarray(sh.array[..., 3:8])

fig, axes = plt.subplots(1, 5, figsize=(15, 3), subplot_kw={"projection": "3d"})
for m, ax in enumerate(axes):
    val = y2[..., m]
    r_plot = np.abs(val)
    ax.plot_surface(
        r_plot * np.asarray(x), r_plot * np.asarray(y), r_plot * np.asarray(z),
        facecolors=plt.cm.coolwarm((val - val.min()) / (val.max() - val.min() + 1e-12)),
        rstride=1, cstride=1, antialiased=False, linewidth=0,
    )
    ax.set_title(f"$Y_2^{{m={m-2}}}$")
    ax.set_axis_off()
plt.tight_layout(); plt.show()

## 12.11 Cell 5 — The promise: spherical harmonics rotate by a fixed matrix

✅ **Checkpoint A**

Stack the $2\ell+1$ spherical harmonics at fixed $\ell$ into a column vector
$$Y_\ell(\hat r) \;=\; \big(\, Y_\ell^{-\ell}(\hat r),\; Y_\ell^{-\ell+1}(\hat r),\; \ldots,\; Y_\ell^{+\ell}(\hat r) \,\big)^{\!\top}.$$
The central identity of this chapter is that rotating the *argument* and rotating the *output vector* give the same result, with the second rotation expressed by a matrix that depends only on $R$ and $\ell$:
$$\boxed{\;Y_\ell(R\,\hat r) \;=\; D^\ell(R)\,Y_\ell(\hat r)\;}\tag{12.1}$$
The matrix $D^\ell(R)$ is called the **Wigner-D matrix**. It has size $(2\ell+1)\times(2\ell+1)$ and is itself a rotation, $D^\ell(R)^\top D^\ell(R) = I$. We never need a closed form: `e3nn-jax` constructs $D^\ell(R)$ from $R$ for us.

> **Reading (12.1) as a representation.** A *representation* of a group $G$ on a vector space $V$ is a rule $\rho : G \to \mathrm{GL}(V)$ that respects multiplication, $\rho(R_1 R_2) = \rho(R_1)\rho(R_2)$. Equation (12.1) says that $V_\ell := \mathrm{span}\{Y_\ell^m\}$ is such a $V$ for $G = \mathrm{SO}(3)$ with $\rho = D^\ell$. The representation is *irreducible* — there is no nontrivial subspace of $V_\ell$ closed under all $D^\ell(R)$ — because a single $\ell$ value is a definite total angular momentum quantum number. The labels `0e, 1o, 2e, …` from §12.12 enumerate exactly the irreducible representations of $\mathrm{O}(3)$.

> **Why this is the foundation.** Every other equivariant operation in this chapter is built by combining irreducible blocks, and the combination rules use $D^\ell(R)$ as their atomic ingredient. The closed transformation rule (12.1) is the technical reason equivariant networks exist.

The cell below picks 16 random unit vectors and a random rotation $R$, computes both sides of (12.1) numerically, and confirms they agree to $\sim 10^{-15}$.

![commutative diagram for spherical harmonic equivariance.](images/05_equivariance_diagram.png)


In [ ]:
# Random unit vector and a random rotation
k1, k2 = jax.random.split(jax.random.PRNGKey(7))
rhat = jax.random.normal(k1, (16, 3))
rhat = rhat / jnp.linalg.norm(rhat, axis=-1, keepdims=True)
rhat = e3nn.IrrepsArray("1o", rhat)         # tag as 1o IrrepsArray
R = e3nn.rand_matrix(k2)

# Method 1: SH at rotated coordinates
lmax = 3
Y_at_Rr = e3nn.spherical_harmonics(list(range(1, lmax + 1)), rhat.transform_by_matrix(R), normalize=True)

# Method 2: SH at original coords, then library applies D^l(R)
Y_at_r = e3nn.spherical_harmonics(list(range(1, lmax + 1)), rhat, normalize=True)
Y_rotated = Y_at_r.transform_by_matrix(R)

diff = jnp.max(jnp.abs(Y_at_Rr.array - Y_rotated.array))
print(f"max |Y(R r) - D(R) Y(r)| = {diff:.2e}")
assert diff < 1e-8, "equivariance violated. something is wrong"
print("✅ Checkpoint A: equivariance of spherical harmonics holds")

## 12.12 Cell 6 — Type tags: the `Irreps` declaration

In a generic neural network a feature vector $h \in \mathbb{R}^n$ is a flat array; the network has no information about how the entries should transform. An equivariant network attaches a **type tag** to every block of entries, declaring how that block transforms under $\mathrm{O}(3)$.

The tags are exactly the irreducible representations of $\mathrm{O}(3)$, labelled by $\ell$ and parity $p \in \{+,-\}$. The `e3nn` notation is

| Tag | $\ell$, parity | dim | physical example |
|---|---|---|---|
| `0e` | 0, even | 1 | scalar (energy, charge, temperature) |
| `0o` | 0, odd | 1 | pseudoscalar (helicity) |
| `1o` | 1, odd | 3 | polar vector (position, momentum, force) |
| `1e` | 1, even | 3 | axial vector (angular momentum, magnetic field) |
| `2e` | 2, even | 5 | symmetric rank-2 tensor (stress, polarisability) |
| `2o` | 2, odd | 5 | parity-odd rank-2 tensor |

These tags are *complete*: every finite-dimensional representation of $\mathrm{O}(3)$ decomposes into a direct sum of them.

A typed feature is declared by an irreps string such as
$$\texttt{"32x0e + 8x1o + 8x2e"},$$
which means "thirty-two scalars, eight polar vectors, eight rank-2 tensors". The total dimension is
$$32\cdot 1 + 8\cdot 3 + 8\cdot 5 \;=\; 96.$$
A plain $\mathbb{R}^{96}$ feature would carry the same number of components but no information about how the components transform. Once the tags are present, every layer in the network applies the appropriate rotation rule to each block separately:

* On a `0e` block, rotation acts as the identity.
* On a `1o` block, rotation acts as $R$ itself.
* On an `\ell e` or `\ell o` block, rotation acts as $D^\ell(R)$.

Mixing different blocks linearly across types would in general break equivariance, so equivariant linear layers act *block-diagonally* in the irrep basis. Cross-type interaction is the job of the tensor product (§12.13).

Think of `Irreps` as the equivariant analogue of a static type system: just as `int + float` is forbidden until you cast, mixing `0e` and `1o` requires going through a typed operation that produces a well-defined output type.

![layout of a 96-number feature with type tags.](images/06_irreps_layout.png)


In [ ]:
irreps = e3nn.Irreps("32x0e + 8x1o + 8x2e")
print(f"irreps          : {irreps}")
print(f"total dim       : {irreps.dim}        (= 32*1 + 8*3 + 8*5 = 96)")
print(f"lmax            : {irreps.lmax}")
print(f"num irreps      : {len(irreps)}")
print(f"slices          : {irreps.slices()}")

# Compare to a non-equivariant view: 96 raw scalars (no symmetry information)
vanilla = e3nn.Irreps("96x0e")
print(f"\nA vanilla 96-dim feature would be: {vanilla}")
print("Same dim, but the equivariant declaration above carries 8 vectors and 8 rank-2 channels with known rotation behavior.")

## 12.13 Cell 7 — The equivariant tensor product

✅ **Checkpoint B**

Equivariant linear layers cannot mix different irrep types. Cross-type interaction comes from the **tensor product**, written $\otimes$. Given two equivariant features $a$ and $b$ with irrep types $\ell_1$ and $\ell_2$, the tensor product $a\otimes b$ produces a new equivariant feature whose irrep content is fixed by the **Clebsch–Gordan rule**:
$$\boxed{\;\ell_1 \otimes \ell_2 \;=\; \bigoplus_{\ell = |\ell_1 - \ell_2|}^{\ell_1 + \ell_2} \ell\;}\tag{12.2}$$
with parities multiplying as $p_1 \cdot p_2$. The output dimensions sum to $(2\ell_1+1)(2\ell_2+1)$, matching the dimension of the underlying real tensor product.

Specialising to two polar vectors ($\ell_1 = \ell_2 = 1$, both parity-odd):
$$\mathbf{1o} \;\otimes\; \mathbf{1o} \;=\; \mathbf{0e} \;\oplus\; \mathbf{1e} \;\oplus\; \mathbf{2e}.$$
The output is the well-known decomposition of an outer product of two 3-vectors $a_i b_j$ into trace, antisymmetric part and traceless symmetric part:

| Block | Component formula | Output type | Numbers |
|---|---|---|---|
| **Dot product** | $\vec a \cdot \vec b \;=\; \sum_{i=1}^{3} a_i b_i$ | scalar `0e` | 1 |
| **Cross product** | $(\vec a \times \vec b)_k \;=\; \sum_{ij} \varepsilon_{kij}\,a_i b_j$ | axial vector `1e` | 3 |
| **Traceless symmetric** | $T_{ij} \;=\; \tfrac{1}{2}(a_i b_j + a_j b_i) \;-\; \tfrac{1}{3}(\vec a \cdot \vec b)\,\delta_{ij}$ | rank-2 `2e` | 5 |

Total: $1+3+5 = 9$, matching the nine independent components of $a_i b_j$.

> **Symbols.** $\delta_{ij}$ is the Kronecker delta ($1$ if $i=j$, else $0$). $\varepsilon_{kij}$ is the Levi-Civita symbol ($+1$/$-1$ on even/odd permutations of $(1,2,3)$, $0$ otherwise). The dot, cross and traceless-symmetric pieces are the *only* rotation-respecting bilinear maps $\mathbb{R}^3 \times \mathbb{R}^3 \to V_\ell$, a consequence of (12.2).
>
> **Parity.** Both `1o` factors are parity-odd, so each output block is parity-even (`0e`, `1e`, `2e`). For mixed parities the same rule applies: `1e` ⊗ `1o` = `0o ⊕ 1o ⊕ 2o`, and so on.

Two more symbols, used everywhere from now on.

* $\otimes$ — equivariant tensor product, computed by `e3nn.tensor_product`. Returns an `IrrepsArray` whose irreps follow (12.2).
* $\oplus$ — direct sum, i.e. concatenation of irrep blocks, computed by `e3nn.concatenate`.

`e3nn.tensor_product(a, b)` returns *all* allowed output blocks at once. The cell below verifies that the `0e` channel really is the dot product, up to a normalisation $1/\sqrt{3}$ that follows from `e3nn-jax`'s *component-normalised* Clebsch–Gordan convention: the coupling coefficients are scaled so that, when the inputs $a, b$ have i.i.d. unit-variance components, the components of $a\otimes b$ also have unit variance. This factor is bookkeeping, not physics.

![tensor product of two vectors decomposed into 0e, 1e, 2e.](images/07_tensor_product.png)


In [ ]:
k1, k2 = jax.random.split(jax.random.PRNGKey(11))
a_data = jax.random.normal(k1, (3,))
b_data = jax.random.normal(k2, (3,))

a = e3nn.IrrepsArray("1o", a_data)
b = e3nn.IrrepsArray("1o", b_data)

ab = e3nn.tensor_product(a, b)
print(f"a       irreps : {a.irreps}")
print(f"a.shape         : {a.array.shape}")
print(f"a ⊗ b   irreps : {ab.irreps}")
print(f"(a ⊗ b).shape   : {ab.array.shape}  (= 1 + 3 + 5 = 9 numbers)")

# Slice into the three irrep blocks
scalar = ab.array[..., 0:1]   # 0e block (1 value)
vector = ab.array[..., 1:4]   # 1e block (3 values, the cross product up to sign/norm)
tensor = ab.array[..., 4:9]   # 2e block (5 values)

dot = jnp.dot(a_data, b_data)
cross = jnp.cross(a_data, b_data)

print(f"\n0e channel        : {scalar.squeeze():.6f}")
print(f"a · b / sqrt(3)   : {dot / jnp.sqrt(3.0):.6f}")
ratio_0e = scalar.squeeze() / (dot / jnp.sqrt(3.0))
print(f"ratio              : {ratio_0e:.6f}  (should be 1.0)")
assert jnp.abs(ratio_0e - 1.0) < 1e-6, "0e channel does not match (a·b)/sqrt(3)"
print("\n✅ Checkpoint B: 0e channel of (a ⊗ b) equals (a·b)/sqrt(3) as expected")

# Quick filter API demo
scalar_only = e3nn.tensor_product(a, b, filter_ir_out=[e3nn.Irrep("0e")])
print(f"\nfilter_ir_out=[0e] gives irreps: {scalar_only.irreps}")

## 12.14 Cell 8 — The Tetris benchmark

The dataset reproduced here is from Thomas et al. (2018), Section 4.1. It has been the canonical equivariance micro-benchmark for the past five years for three reasons.

* **Eight classes.** Few enough to enumerate, large enough that random guessing is a clear $1/8 = 12.5\%$ baseline.
* **Discrete integer geometry.** Every shape is fixed by 4 atomic positions on a 3D integer grid, so visual identification is unambiguous.
* **Includes a chiral pair.** Two of the eight shapes are mirror images of each other. Any *reflection-invariant* model — for instance a graph network whose only edge feature is the bond length $r_{ij}$ — is forced by parity to assign mirror images identical scores. Distinguishing them probes the parity-odd channels of the architecture, not just rotation.

| Label | Name | Description |
|---|---|---|
| 0 | `chiral_1` | chiral shape, one specific handedness |
| 1 | `chiral_2` | mirror image of `chiral_1` |
| 2 | `square` | flat $2 \times 2$ square |
| 3 | `line` | 4 atoms in a line |
| 4 | `corner` | three legs from a corner |
| 5 | `L` | L-shape |
| 6 | `T` | T-shape |
| 7 | `zigzag` | zigzag |

The discriminator that the equivariant network will eventually learn for the chiral pair is a **parity-odd scalar** channel `0o` in the last layer, introduced in §12.16. Under reflection a `0o` value flips sign, providing the single bit of information needed to tell `chiral_1` from `chiral_2`.

![the 8 shapes, with the mirror pair circled.](images/08_tetris_shapes.png)


In [ ]:
shape_names = ["chiral_1", "chiral_2", "square", "line", "corner", "L", "T", "zigzag"]
pos = jnp.array([
    [[0, 0, 0], [0, 0, 1], [1, 0, 0], [1, 1, 0]],   # chiral_shape_1
    [[1, 1, 1], [1, 1, 2], [2, 1, 1], [2, 0, 1]],   # chiral_shape_2  (mirror)
    [[0, 0, 0], [1, 0, 0], [0, 1, 0], [1, 1, 0]],   # square
    [[0, 0, 0], [0, 0, 1], [0, 0, 2], [0, 0, 3]],   # line
    [[0, 0, 0], [0, 0, 1], [0, 1, 0], [1, 0, 0]],   # corner
    [[0, 0, 0], [0, 0, 1], [0, 0, 2], [0, 1, 0]],   # L
    [[0, 0, 0], [0, 0, 1], [0, 0, 2], [0, 1, 1]],   # T
    [[0, 0, 0], [1, 0, 0], [1, 1, 0], [2, 1, 0]],   # zigzag
], dtype=jnp.float64)
labels = jnp.arange(8)
print(f"pos shape   : {pos.shape}  (8 shapes, 4 atoms, 3 coords)")

fig = plt.figure(figsize=(14, 7))
for i, name in enumerate(shape_names):
    ax = fig.add_subplot(2, 4, i + 1, projection="3d")
    p = np.asarray(pos[i])
    ax.scatter(p[:, 0], p[:, 1], p[:, 2], s=80)
    # draw bonds where atoms are within distance 1.1
    for a_idx in range(4):
        for b_idx in range(a_idx + 1, 4):
            if np.linalg.norm(p[a_idx] - p[b_idx]) < 1.1:
                ax.plot(*zip(p[a_idx], p[b_idx]), color="black", lw=1)
    ax.set_title(name)
    ax.set_xlim(-0.5, 2.5); ax.set_ylim(-0.5, 2.5); ax.set_zlim(-0.5, 3.5)
plt.tight_layout(); plt.show()

For training, each shape is packaged as a graph: nodes are atoms; edges connect atom
pairs within distance $r_c = 1.1$ (just enough to link adjacent grid points). The eight
per-shape graphs are stacked into a single `jraph.GraphsTuple` so that one forward pass
processes the whole batch.

> **Aside (pre-built, no new ideas).** The helper `make_tetris_graphs()` below is plain
> graph-batching boilerplate — `radius_graph` to build edges, then `jraph.batch` to stack.
> It is provided complete; you do **not** need to modify it. Run it and move on to the
> physics in §12.15.
>
> **※ Footnote — the L11 Fact 2, corrected.** When you reproduce the L11 SchNet baseline,
> recall that at the short cutoff $r_c = 1.1$ several Tetris shapes collapse to the *same*
> path graph once you keep only nearest-neighbour bonds. There are **five** such
> path-graph-isomorphic shapes — `chiral_1`, `chiral_2`, `line`, `L`, `zigzag` — all of
> which look like a 4-node chain to a distance-only model. That leaves only **3
> distinguishable classes out of 8** at this cutoff, so a purely distance-based message
> caps out at $3/8 = 37.5\%$ accuracy (not the "4 shapes / 50%" stated in an earlier draft
> of L11). The equivariant message defeats this because $Y_\ell(\hat r_{ij})$ carries the
> *angle* of each bond, which the five chains do **not** share.

In [ ]:
def make_tetris_graphs() -> jraph.GraphsTuple:
    graphs = []
    for p, lab in zip(pos, labels):
        senders, receivers = e3nn.radius_graph(p, 1.1)
        graphs.append(jraph.GraphsTuple(
            nodes=p.reshape((4, 3)),
            edges=None,
            globals=lab[None],
            senders=senders,
            receivers=receivers,
            n_node=jnp.array([4]),
            n_edge=jnp.array([len(senders)]),
        ))
    return jraph.batch(graphs)

graphs = make_tetris_graphs()
print(f"total nodes : {graphs.n_node.sum()}")
print(f"total edges : {graphs.n_edge.sum()}")
print(f"labels      : {graphs.globals}")

## 12.15 Cell 9 — From SchNet to the equivariant message

The MPNN template inherited from L11 is unchanged: every layer is *message → aggregate → update*. The only change is that every operation now acts on `IrrepsArray`s rather than plain $\mathbb{R}^n$ vectors. The substitution table is short.

| Component | SchNet (L11) | Equivariant (this chapter) |
|---|---|---|
| Node feature $h_i$ | scalar vector $\mathbb{R}^d$ | `IrrepsArray`, e.g. `32x0e + 8x1o + 8x2e` (§12.12) |
| Edge feature | $\mathrm{RBF}(r_{ij})$ — depends on $\|\vec r_{ij}\|$ only | $Y_\ell(\hat r_{ij})$ — depends on the *direction* (§§12.10–12.11) |
| Combine $h_j$ with edge | elementwise gate $(W h_j) \odot \mathrm{MLP}(\mathrm{RBF})$ | tensor product $h_j \otimes Y_\ell(\hat r_{ij})$ (§12.13) |
| Aggregation | $\sum_{j\in\mathcal{N}(i)}$ | $\sum_{j\in\mathcal{N}(i)}$ — unchanged; sum of irreps of the same type is itself equivariant |
| Update | linear + nonlinearity | irrep-wise linear + scalar nonlinearity |

**What is gained.**

* *Direction.* For $\ell \geq 1$, $Y_\ell(\hat r_{ij})$ encodes the angular geometry of bond $(i,j)$, which $\mathrm{RBF}(r_{ij})$ throws away. This is what fixes the rotation failure of the flat-MLP baseline.
* *Parity.* For odd $\ell$, $Y_\ell$ flips sign under reflection. Carrying any parity-odd channel (`1o`, `3o`, …) through to the output gives the model an internal quantity that distinguishes mirror images. This is what fixes the chirality blindness of SchNet.

**What is preserved.**

* Graph topology. Atoms = nodes; neighbours within cutoff = edges. Identical to L11.
* Three-step MPNN template (Gilmer et al., 2017). Identical to L11.
* Permutation invariance over neighbours. Identical to L11; provided by $\sum$.

**A worked single bond.** Take a node feature $h_j = (s, \vec v)$ of type `1x0e + 1x1o`, a bond direction $\hat r_{ij}$, and use $Y_1(\hat r_{ij}) = \hat r_{ij}$ (up to `e3nn` component normalisation and real-basis sign conventions; see §12.13). Then by (12.2),
$$h_j \otimes Y_1(\hat r_{ij}) \;=\; \underbrace{s\,\hat r_{ij}}_{\text{`1o` block}} \;\oplus\; \underbrace{(\vec v \cdot \hat r_{ij})}_{\text{`0e` block}} \;\oplus\; \underbrace{\vec v \times \hat r_{ij}}_{\text{`1e` block}} \;\oplus\; \underbrace{T(\vec v, \hat r_{ij})}_{\text{`2e` block}}.$$
Every block on the right is a familiar physical quantity. The network learns *coefficients* on each block; the geometry of the block itself is fixed by symmetry.

The next section assembles this message inside a `jraph.GraphNetwork` and stacks two layers.

> **Citation note (SchNet vs DTNN).** SchNet — the distance-only continuous-filter MPNN we are upgrading here — is **Schütt et al., *SchNet: A continuous-filter convolutional neural network for modeling quantum interactions*, NeurIPS 2017 (arXiv:1706.08566)**. This is a *different* paper from the deep tensor neural network (DTNN), **Schütt et al., *Quantum-chemical insights from deep tensor neural networks*, Nat. Commun. 8, 13890 (2017)** — DTNN is an earlier message-passing model on molecular graphs, not SchNet. An earlier draft conflated the two; the References cell at the end lists both correctly.

![SchNet message vs equivariant message](images/bridge_schnet_to_equivariant.png)


## 12.16 Cell 10 — Architecture and training

✅ **Checkpoint C**

The model is two stacked equivariant layers followed by an irrep-shaped readout. We spell out one layer first, then the readout, then the loss.

### 12.16.1 One equivariant layer

Within one molecule, atoms are indexed by $i, j, k, \ldots$.

* $\vec r_i \in \mathbb{R}^3$ is the position of atom $i$.
* $\vec r_{ij} \;=\; \vec r_i - \vec r_j$ is the displacement from sender $j$ to receiver $i$. (This matches the code line `rij = positions[receivers] - positions[senders]`; the choice flips $\hat r_{ij}\mapsto -\hat r_{ij}$ relative to the physics convention $\vec r_j-\vec r_i$, which only changes the sign of odd-$\ell$ spherical harmonics — the network absorbs the sign in its learnable coefficients.)
* $r_{ij} \;=\; |\vec r_{ij}|$ is the bond length, $\hat r_{ij} \;=\; \vec r_{ij} / r_{ij}$ the bond direction.
* $h_i$ is the feature carried by atom $i$, an `IrrepsArray` (a typed bundle of scalars, vectors and tensors).
* $r_c$ is a fixed cutoff distance. We use $r_c = 1.1$, just enough to connect grid-adjacent atoms.
* $\mathcal{N}(i) \;=\; \{\, j \;:\; r_{ij} < r_c \,\}$ is the neighbourhood of atom $i$.

> **Cutoff comparison with L11.** L11's SchNet used a fully-connected graph ($r_c = 4$) because distance-only features cannot distinguish certain isomorphic graphs at small $r_c$. Direction-aware messages already break that degeneracy, so the present model is content with a much shorter cutoff.

For each ordered pair $(i, j)$ with $j \in \mathcal{N}(i)$, the message from sender $j$ to receiver $i$ is
$$m_{ij} \;=\; h_j \;\otimes\; Y_\ell(\hat r_{ij}),$$
with the library stacking $\ell = 1, 2, 3$ together inside the spherical-harmonic factor. The receiver aggregates,
$$M_i \;=\; \sum_{j \in \mathcal{N}(i)} m_{ij},$$
and the aggregated feature is passed through

1. an **equivariant linear layer**, which mixes the multiplicities of each irrep without changing types (block-diagonal in the irrep basis), then
2. a **scalar activation**, a non-linearity applied only on the `0e`/`0o` channels, leaving higher-$\ell$ blocks untouched. A `0e` scalar is parity-trivial, so any pointwise nonlinearity preserves its type (the default `silu` is fine). A `0o` scalar must satisfy $\sigma(-x) = -\sigma(x)$, i.e. the activation must be *odd*, otherwise applying it would silently break parity equivariance. `e3nn.scalar_activation` enforces this constraint automatically and will raise if a non-odd activation is requested for an odd-parity scalar.

> **On the choice of nonlinearity.** A general non-linearity (e.g. ReLU applied component-wise to a `1o` block) destroys equivariance: the result is not even a vector under rotation. Two equivariant alternatives are common: the simplest one (used here) restricts the non-linearity to the scalar channels, and a *gate activation* (used in NequIP, MACE) propagates non-linearity to higher $\ell$ via a learnable scalar gate that multiplies each $\ell$-vector. The simpler version suffices for this benchmark.

Every step is equivariant by construction. Crucially, the network never sees a rotated copy of any training shape.

### 12.16.2 Output head and chirality

The last layer is shaped to output irreps `"0o + 7x0e"` per atom.

* `0o` — a single parity-odd scalar. Under reflection it flips sign. **This is the only feature in the model that distinguishes a left-handed shape from a right-handed shape.**
* `7x0e` — seven ordinary scalars. Six handle the six non-chiral classes; one is paired with the `0o` to form the two chiral logits.

The four atoms' per-atom outputs are summed over atoms within each shape to give one length-8 vector per shape. Write
$$s_o = (\text{`0o` value}), \quad s_e = (\text{first `0e` value}), \quad e_2, \ldots, e_7 = (\text{remaining six `0e` values}).$$
The eight class logits are
$$\text{logits} \;=\; \big[\; +s_o\, s_e,\;\; -s_o\, s_e,\;\; e_2,\;\; e_3,\;\; e_4,\;\; e_5,\;\; e_6,\;\; e_7 \;\big].$$
Slots 0 and 1 — the chiral pair — share the same magnitude $|s_o s_e|$ and opposite sign. Reflecting a shape sends $s_o \mapsto -s_o$ and leaves $s_e$ untouched (it is `0e`), so the two chiral logits swap exactly while the six non-chiral logits are unchanged. Provided that on the chiral inputs the chiral pair dominates the argmax (which is the regime the network learns), the predicted class flips between slots 0 and 1. The other six slots are plain parity-even scalars. This is the *minimal* readout that respects $\mathrm{O}(3)$ and can label all eight classes.

### 12.16.3 Loss and compile time

The loss is softmax cross-entropy with integer labels; the optimiser is Adam at learning rate $10^{-2}$. The first call of the JIT-compiled step takes 30–90 s (XLA compilation); subsequent steps are subsecond. Training caps at 200 steps and usually reaches 100% well before that.

![end-to-end architecture diagram with the chiral output head highlighted.](images/09_architecture.png)


In [ ]:
import time

class Layer(nn.Module):
    target_irreps: str
    denominator: float
    sh_lmax: int = 3

    @nn.compact
    def __call__(self, gr, positions):
        target = e3nn.Irreps(self.target_irreps)

        def update_edge_fn(edge, sender, receiver, glob):
            # `positions` is an IrrepsArray('1o'), so subtraction yields IrrepsArray('1o')
            # which spherical_harmonics accepts directly.
            rij = positions[gr.receivers] - positions[gr.senders]
            sh = e3nn.spherical_harmonics(
                list(range(1, self.sh_lmax + 1)), rij, True,
            )
            return e3nn.concatenate([sender, e3nn.tensor_product(sender, sh)]).regroup()

        def update_node_fn(node, sender_msg, receiver_msg, glob):
            x = receiver_msg / self.denominator
            x = e3nn.flax.Linear(target, name="lin_pre")(x)
            x = e3nn.scalar_activation(x)
            x = e3nn.flax.Linear(target, name="lin_post")(x)
            short = e3nn.flax.Linear(x.irreps, name="shortcut", force_irreps_out=True)(node)
            return short + x

        return jraph.GraphNetwork(update_edge_fn, update_node_fn)(gr)


class TetrisModel(nn.Module):
    @nn.compact
    def __call__(self, gr):
        positions = e3nn.IrrepsArray("1o", gr.nodes)
        gr = gr._replace(nodes=jnp.ones((len(positions), 1)))
        layers = 2 * ["32x0e + 32x0o + 8x1e + 8x1o + 8x2e + 8x2o"] + ["0o + 7x0e"]
        for irreps in layers:
            gr = Layer(irreps, 1.5)(gr, positions)

        # Pool per graph: sum atomic features into shape (num_graphs, 8).
        pred = e3nn.scatter_sum(gr.nodes.array, nel=gr.n_node)

        # The last layer's irreps were "0o + 7x0e". The 8 features per graph are:
        #   pred[:, 0]   = the 0o (parity-odd scalar). It flips sign on reflection.
        #                  This is what tells chiral_1 from chiral_2.
        #   pred[:, 1]   = a 0e magnitude paired with the 0o for the chiral logits.
        #   pred[:, 2:8] = six 0e logits, one for each non-chiral class.
        # We then build 8 class logits in label order
        # [chiral_1, chiral_2, square, line, corner, L, T, zigzag]:
        odd, even1, even2 = pred[:, :1], pred[:, 1:2], pred[:, 2:]
        logits = jnp.concatenate([odd * even1, -odd * even1, even2], axis=1)
        return logits


model = TetrisModel()
opt = optax.adam(0.01)

def loss_fn(params, gr):
    logits = model.apply(params, gr)
    labs = gr.globals
    loss = jnp.mean(optax.softmax_cross_entropy_with_integer_labels(logits, labs))
    return loss, logits

@jax.jit
def step(params, opt_state, gr):
    grads, logits = jax.grad(loss_fn, has_aux=True)(params, gr)
    acc = jnp.mean(jnp.argmax(logits, axis=1) == gr.globals)
    updates, opt_state = opt.update(grads, opt_state)
    return optax.apply_updates(params, updates), opt_state, acc, logits

params = jax.jit(model.init)(jax.random.PRNGKey(SEED), graphs)
opt_state = opt.init(params)

wall = time.perf_counter()
print("JIT-compiling step (this can take 30-90 s on first call)...")
params, opt_state, acc, logits = step(params, opt_state, graphs)
print(f"compile + first step: {time.perf_counter() - wall:.1f} s, initial acc = {100*acc:.0f}%")

loss_hist, acc_hist = [], []
for s in tqdm(range(200)):
    params, opt_state, acc, logits = step(params, opt_state, graphs)
    loss_val, _ = loss_fn(params, graphs)
    loss_hist.append(float(loss_val))
    acc_hist.append(float(acc))
    if acc == 1.0:
        print(f"\nReached 100% accuracy at step {s}")
        break

print(f"final training accuracy: {100*acc_hist[-1]:.0f}%")

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot(loss_hist); axes[0].set_xlabel("step"); axes[0].set_ylabel("cross-entropy loss")
axes[1].plot(acc_hist); axes[1].set_xlabel("step"); axes[1].set_ylabel("accuracy")
axes[1].set_ylim(0, 1.05)
plt.tight_layout(); plt.show()
print("\n[OK] Checkpoint C: equivariant network classifies all 8 Tetris shapes, including the chiral pair.")

## 12.17 Cell 11 — The rotation test

The equivariant model trained on each shape exactly once, with no rotated copies. Two experiments now make the data-efficiency claim concrete.

**Part 1. Equivariance test.** Take all 8 shapes, rotate by 100 random matrices drawn uniformly from $\mathrm{SO}(3)$, classify each rotated copy. Expected accuracy: 100%, exactly. Rotation does not change the model's output by construction; the test verifies that the implementation honours that construction to floating-point precision.

**Part 2. Vanilla-MLP baseline.** Train a 3-layer MLP on the eight flat-coordinate inputs of shape $(4\times 3 = 12)$. Run the same rotation test. Expected accuracy:
$$\frac{1}{8} \;\approx\; 12.5\%,$$
the random-guessing rate. The MLP has no built-in notion of rotation; with only one training example per class, it has no statistical opportunity to learn one either. (This reproduces the L11 finding for completeness, this time alongside an equivariant network on identical axes.)

The gap between the two curves is the data-efficiency claim in miniature. To match the equivariant model's rotation behaviour with augmentation alone, the vanilla MLP would need on the order of hundreds to thousands of rotated copies per shape — and even then it would only ever be approximately rotation-invariant on rotations seen near training time.

![accuracy comparison bar chart, equivariant vs vanilla MLP.](images/10_accuracy_comparison.png)


In [ ]:
# Part 2: vanilla MLP baseline trained on flat coordinates
X_train = pos.reshape(8, 12)             # flatten 4 atoms × 3 coords
y_train = labels

class MLP(nn.Module):
    @nn.compact
    def __call__(self, x):
        x = nn.Dense(128)(x); x = nn.silu(x)
        x = nn.Dense(128)(x); x = nn.silu(x)
        return nn.Dense(8)(x)

mlp = MLP()
mlp_params = mlp.init(jax.random.PRNGKey(0), X_train)
mlp_opt = optax.adam(0.01)
mlp_state = mlp_opt.init(mlp_params)

@jax.jit
def mlp_step(params, state, X, y):
    def loss_fn(p):
        logits = mlp.apply(p, X)
        return jnp.mean(optax.softmax_cross_entropy_with_integer_labels(logits, y)), logits
    grads, logits = jax.grad(loss_fn, has_aux=True)(params)
    updates, state = mlp_opt.update(grads, state)
    return optax.apply_updates(params, updates), state, jnp.mean(jnp.argmax(logits, axis=1) == y)

train_acc_hist, rot_acc_hist = [], []
for s in tqdm(range(1000)):
    mlp_params, mlp_state, train_acc = mlp_step(mlp_params, mlp_state, X_train, y_train)
    if s % 25 == 0:
        # rotated-test accuracy: 100 random rotations of the training shapes
        rks = jax.random.split(jax.random.PRNGKey(123 + s), 100)
        accs = []
        for k in rks:
            R = e3nn.rand_matrix(k)
            X_rot = (pos @ R.T).reshape(8, 12)
            preds = jnp.argmax(mlp.apply(mlp_params, X_rot), axis=1)
            accs.append(float(jnp.mean(preds == labels)))
        rot_acc_hist.append((s, np.mean(accs)))
    train_acc_hist.append((s, float(train_acc)))

ts, tas = zip(*train_acc_hist)
rs, ras = zip(*rot_acc_hist)
plt.figure(figsize=(7, 3.5))
plt.plot(ts, tas, label="vanilla MLP, train acc (8 shapes)", color="C0")
plt.plot(rs, ras, label="vanilla MLP, rotated-test acc", color="C3")
plt.axhline(1.0, color="C2", ls="--", lw=1, label="equivariant net (rotated-test)")
plt.axhline(1/8, color="gray", ls=":", lw=1, label="random (1/8)")
plt.xlabel("step"); plt.ylabel("accuracy"); plt.ylim(0, 1.05)
plt.legend(loc="center right", fontsize=8); plt.tight_layout(); plt.show()

print(f"vanilla MLP final train acc       : {train_acc_hist[-1][1]*100:.0f}%")
print(f"vanilla MLP final rotated-test acc: {rot_acc_hist[-1][1]*100:.0f}%   (≈ random)")
print("\nThe MLP memorizes the 8 training shapes. It has no idea what to do with rotated copies. That is exactly the gap equivariance closes.")

---

# 12.A Real molecules: energy regression on rMD17 aspirin

Everything so far lived on the **toy** Tetris benchmark: integer coordinates, eight
hand-built shapes, a classification label. That was the right place to *prove* the
machinery (exact equivariance to $10^{-15}$, chirality from a `0o` channel). Now we point
the **same** equivariant message-passing network at a **real molecule** and ask it to do
the job these networks were actually invented for: **predict the potential energy from the
atomic coordinates, in physical units.**

## The dataset: rMD17

**rMD17** (*revised* MD17; Christensen & von Lilienfeld, *Mach. Learn.: Sci. Technol.* **1**,
045018, 2020, [DOI:10.1088/2632-2153/abba6f](https://doi.org/10.1088/2632-2153/abba6f)) is a
recomputed, tighter-converged version of the classic MD17 benchmark. For each of ten small
molecules it provides $\sim$100k snapshots from an *ab initio* molecular-dynamics trajectory,
each labelled with a DFT (PBE/def2-SVP) **total energy** and the **forces on every atom**.

We use **aspirin** (C$_9$H$_8$O$_4$, $N=21$ atoms). Units:

| Quantity | Symbol | Unit |
|---|---|---|
| atomic positions | $\vec r_i$ | Å (ångström) |
| total energy | $E$ | eV (electron-volt) |
| force on atom $i$ | $\vec F_i = -\partial E/\partial \vec r_i$ | eV / Å |

This is a tiny problem by ML-potential standards — we deliberately take a **train/test
subset (950 / 50 snapshots)** so the whole thing trains in a few minutes on a Colab T4. The
goal is pedagogical: experience real units, a real DFT ground truth, and the
"toy $\to$ real" performance gap — not to set a benchmark record.

> **Physicist's reading.** $E(\{\vec r_i\})$ is a *potential energy surface*. A rotation or
> reflection of the whole molecule cannot change its energy (the Hamiltonian is
> $\mathrm{O}(3)$-symmetric), so $E$ must be an $\mathrm{O}(3)$-**invariant** (a pure `0e`
> scalar) — exactly what the readout of our network produces. The forces are minus the
> gradient of that scalar, so they come out as `1o` vectors **for free** via `jax.grad`.

In [ ]:
# --- Load the rMD17 aspirin trajectory ---------------------------------------
# rMD17 ships as one .npz per molecule. We pull aspirin directly from the public
# figshare mirror (a few MB). If the download is blocked in your environment, the
# `mace-torch`/`ase` route is given in the comment at the bottom of this cell.
import os, urllib.request
import numpy as np

# Public figshare record 12672038 ("Revised MD17 dataset (rMD17)").
# File: rmd17_aspirin.npz  (positions/energies/forces for ~100k frames).
RMD17_ASPIRIN_URL = (
    "https://figshare.com/ndownloader/files/23950376"  # rmd17_aspirin.npz
)
LOCAL = "rmd17_aspirin.npz"

if not os.path.exists(LOCAL):
    print("downloading rMD17 aspirin (~5 MB) ...")
    urllib.request.urlretrieve(RMD17_ASPIRIN_URL, LOCAL)

raw = np.load(LOCAL)
print("keys in npz:", list(raw.keys()))

# rMD17 field names: 'nuclear_charges' (Z), 'coords' (Å), 'energies' (kcal/mol),
# 'forces' (kcal/mol/Å). We convert energies & forces to eV.
KCAL_PER_MOL_TO_EV = 0.0433641153
Z_atom   = np.asarray(raw["nuclear_charges"]).astype(np.int32)        # (N_atoms,)
coords   = np.asarray(raw["coords"]).astype(np.float64)               # (N_frames, N, 3) Å
energies = np.asarray(raw["energies"]).astype(np.float64) * KCAL_PER_MOL_TO_EV   # eV
forces   = np.asarray(raw["forces"]).astype(np.float64)  * KCAL_PER_MOL_TO_EV    # eV/Å

N_atoms = coords.shape[1]
print(f"atoms per molecule N = {N_atoms}")
print(f"nuclear charges Z    : {Z_atom.tolist()}")
print(f"coords  shape        : {coords.shape}   (frames, atoms, 3)  [Å]")
print(f"energies shape       : {energies.shape}  [eV]")
print(f"forces  shape        : {forces.shape}    [eV/Å]")

# --- Train / test subset ----------------------------------------------------
rng = np.random.default_rng(42)
N_TRAIN, N_TEST = 950, 50
perm = rng.permutation(coords.shape[0])
idx_train, idx_test = perm[:N_TRAIN], perm[N_TRAIN:N_TRAIN + N_TEST]

# Center each frame at its centroid (translation invariance: E is unchanged).
def center(x):
    return x - x.mean(axis=1, keepdims=True)

pos_train, pos_test = center(coords[idx_train]), center(coords[idx_test])
E_train,  E_test    = energies[idx_train], energies[idx_test]
F_train,  F_test    = forces[idx_train],   forces[idx_test]

# Energies span a huge absolute offset; regress the *shifted* energy so the target
# is O(1) eV. We subtract the training-set mean and report MAE on the same scale.
E_mean = E_train.mean()
E_train_c, E_test_c = E_train - E_mean, E_test - E_mean
print(f"\ntrain frames {N_TRAIN}, test frames {N_TEST}")
print(f"energy mean (subtracted offset): {E_mean:.3f} eV")
print(f"train energy spread (std)      : {E_train_c.std():.4f} eV")

# --- Alternative loaders (if the figshare URL is unavailable) ----------------
# (1) via ASE+schnetpack mirror, or (2) via the `mace-torch` data utilities:
#       !pip install mace-torch ase
#       from mace.data import ... ; or use ase.io.read on the rMD17 extxyz files.
# Either route yields the same (Z, coords[Å], energies[eV], forces[eV/Å]) arrays.

In [ ]:
# --- Build a fixed-topology molecular graph for each frame -------------------
# Aspirin has a fixed atom set, so we use a single radius graph (r_cut = 3.0 Å,
# captures covalent + close non-bonded neighbours) shared across frames; only the
# node positions change frame to frame. Atom identity enters through a learned
# embedding of the nuclear charge Z (one-hot over the distinct elements present).
import jax
import jax.numpy as jnp
import e3nn_jax as e3nn
import jraph

R_CUT = 3.0  # Å

# Distinct elements in aspirin -> small embedding table.
elements = np.unique(Z_atom)                       # e.g. [1, 6, 8] = H, C, O
z_to_type = {int(z): t for t, z in enumerate(elements)}
atom_type = np.array([z_to_type[int(z)] for z in Z_atom], dtype=np.int32)
n_species = len(elements)
print(f"distinct elements: {elements.tolist()}  ->  {n_species} species")

# A representative frame defines the connectivity (aspirin is a rigid covalent
# skeleton; using one reference frame's edges is standard for single-molecule MD17).
ref = pos_train[0]
senders, receivers = e3nn.radius_graph(jnp.asarray(ref), R_CUT)
senders, receivers = np.asarray(senders), np.asarray(receivers)
print(f"edges at r_cut={R_CUT} Å: {len(senders)}  (avg degree {len(senders)/N_atoms:.1f})")

def frame_to_graph(positions_3d: np.ndarray) -> jraph.GraphsTuple:
    """One frame (N,3) -> a jraph graph with one-hot species on the nodes."""
    onehot = np.eye(n_species, dtype=np.float64)[atom_type]   # (N, n_species)
    return jraph.GraphsTuple(
        nodes={"pos": jnp.asarray(positions_3d), "species": jnp.asarray(onehot)},
        edges=None,
        globals=None,
        senders=jnp.asarray(senders),
        receivers=jnp.asarray(receivers),
        n_node=jnp.array([positions_3d.shape[0]]),
        n_edge=jnp.array([len(senders)]),
    )

g0 = frame_to_graph(pos_train[0])
print("node pos shape    :", g0.nodes["pos"].shape, " (N, 3) [Å]")
print("node species shape:", g0.nodes["species"].shape, f" (N, {n_species}) one-hot")
print("senders/receivers :", g0.senders.shape, g0.receivers.shape)

In [ ]:
# --- TFN energy model: equivariant message passing -> invariant energy -------
# Same recipe as the Tetris model, with three changes for a real PES:
#   (1) nodes start from a learned scalar embedding of the element (0e only),
#   (2) the message uses a radial MLP on |r_ij| (Bessel-like RBF) gating the
#       spherical-harmonic tensor product -- this is the SchNet radial filter,
#       now multiplying a *direction-aware* Y_l (the key upgrade of this lecture),
#   (3) the readout is a single 0e scalar per atom; summed over atoms -> energy.
import flax.linen as fnn
import optax
from functools import partial

HIDDEN = "32x0e + 16x1o + 8x2e"   # hidden irreps (scalars + vectors + rank-2)
SH_LMAX = 2
NUM_LAYERS = 2
NUM_RBF = 8

def rbf(r, num=NUM_RBF, r_cut=R_CUT):
    """Smooth radial basis: num Gaussians on [0, r_cut] with a cosine cutoff."""
    centers = jnp.linspace(0.0, r_cut, num)
    widths = (r_cut / num)
    g = jnp.exp(-((r[..., None] - centers) ** 2) / (2 * widths ** 2))
    envelope = 0.5 * (jnp.cos(jnp.pi * jnp.clip(r, 0, r_cut) / r_cut) + 1.0)
    return g * envelope[..., None]                       # (n_edge, num)

class EnergyLayer(fnn.Module):
    target_irreps: str
    @fnn.compact
    def __call__(self, node_feat, pos, senders, receivers):
        rij = pos[receivers] - pos[senders]              # (n_edge, 3) [Å]
        r = jnp.linalg.norm(rij, axis=-1)                # (n_edge,)
        rhat = e3nn.IrrepsArray("1o", rij / (r[:, None] + 1e-9))
        sh = e3nn.spherical_harmonics(list(range(0, SH_LMAX + 1)), rhat, normalize=True)
        # radial weights modulate each path of the tensor product (SchNet filter):
        w = fnn.Dense(8)(rbf(r)); w = fnn.silu(w)        # (n_edge, 8) learned radial
        msg = e3nn.tensor_product(node_feat[senders], sh)
        # broadcast a learned scalar gate from the radial MLP onto the message scalars:
        msg = msg * (1.0 + w[:, :1])
        # aggregate messages on receivers (permutation-invariant sum):
        agg = e3nn.scatter_sum(msg, dst=receivers, output_size=node_feat.shape[0])
        agg = agg / 4.0                                  # rough avg-degree normalizer
        x = e3nn.flax.Linear(e3nn.Irreps(self.target_irreps))(agg)
        x = e3nn.scalar_activation(x)
        x = e3nn.flax.Linear(e3nn.Irreps(self.target_irreps))(x)
        short = e3nn.flax.Linear(x.irreps, force_irreps_out=True)(node_feat)
        return short + x

class EnergyTFN(fnn.Module):
    @fnn.compact
    def __call__(self, graph):
        pos = graph.nodes["pos"]
        species = graph.nodes["species"]                 # (N, n_species) one-hot
        # initial node features: learned scalar embedding of the element (0e only)
        h0 = fnn.Dense(32)(species)                      # (N, 32) scalars
        node_feat = e3nn.IrrepsArray("32x0e", h0)
        for _ in range(NUM_LAYERS):
            node_feat = EnergyLayer(HIDDEN)(
                node_feat, pos, graph.senders, graph.receivers)
        # readout: one 0e scalar per atom (the atomic energy contribution)
        atom_E = e3nn.flax.Linear("1x0e")(node_feat).array[:, 0]   # (N,)
        return e3nn.scatter_sum(atom_E, nel=graph.n_node)[0]       # scalar total E [eV]

model_E = EnergyTFN()
key = jax.random.PRNGKey(42)
params_E = model_E.init(key, frame_to_graph(pos_train[0]))
n_params = sum(p.size for p in jax.tree_util.tree_leaves(params_E))
print(f"energy model parameters: {n_params:,}")
print(f"E(frame 0) at init     : {model_E.apply(params_E, frame_to_graph(pos_train[0])):.4f} eV  (untrained)")

In [ ]:
# --- Train the energy model (Checkpoint D) ----------------------------------
# Loss = mean-squared error on the (offset-subtracted) total energy, in eV^2.
# ~10 epochs over 950 frames; runs in a few minutes on a Colab T4. The first
# jitted step pays the XLA compile cost (30-90 s); later steps are fast.
from tqdm.auto import tqdm

pos_train_j = jnp.asarray(pos_train)        # (N_TRAIN, N, 3)
E_train_j   = jnp.asarray(E_train_c)        # (N_TRAIN,)
pos_test_j  = jnp.asarray(pos_test)
E_test_j    = jnp.asarray(E_test_c)

def energy_of(params, positions_3d):
    return model_E.apply(params, frame_to_graph_jax(positions_3d))

# jit-friendly graph builder (static connectivity captured from globals above)
senders_j, receivers_j = jnp.asarray(senders), jnp.asarray(receivers)
onehot_j = jnp.asarray(np.eye(n_species)[atom_type])
def frame_to_graph_jax(positions_3d):
    return jraph.GraphsTuple(
        nodes={"pos": positions_3d, "species": onehot_j},
        edges=None, globals=None,
        senders=senders_j, receivers=receivers_j,
        n_node=jnp.array([positions_3d.shape[0]]),
        n_edge=jnp.array([senders_j.shape[0]]),
    )

def mse_loss(params, positions_3d, E_target):
    E_pred = energy_of(params, positions_3d)
    return (E_pred - E_target) ** 2

@jax.jit
def batch_loss(params, P, Et):
    return jnp.mean(jax.vmap(mse_loss, in_axes=(None, 0, 0))(params, P, Et))

opt = optax.adam(5e-3)
opt_state = opt.init(params_E)

@jax.jit
def train_step(params, opt_state, P, Et):
    loss, grads = jax.value_and_grad(batch_loss)(params, P, Et)
    updates, opt_state = opt.update(grads, opt_state)
    return optax.apply_updates(params, updates), opt_state, loss

EPOCHS, BATCH = 12, 32
n_batches = N_TRAIN // BATCH
hist = []
for ep in range(EPOCHS):
    p = np.random.default_rng(ep).permutation(N_TRAIN)
    for b in range(n_batches):
        sel = p[b * BATCH:(b + 1) * BATCH]
        params_E, opt_state, loss = train_step(
            params_E, opt_state, pos_train_j[sel], E_train_j[sel])
    # eval: energy MAE in eV on the held-out test set
    E_pred_test = jax.vmap(lambda x: energy_of(params_E, x))(pos_test_j)
    mae = float(jnp.mean(jnp.abs(E_pred_test - E_test_j)))
    hist.append(mae)
    print(f"epoch {ep+1:2d}/{EPOCHS}  train MSE {float(loss):.4e} eV^2   test energy MAE {mae:.4f} eV")

import matplotlib.pyplot as plt
plt.figure(figsize=(6, 3))
plt.plot(range(1, EPOCHS + 1), hist, "o-")
plt.xlabel("epoch"); plt.ylabel("test energy MAE [eV]")
plt.title("rMD17 aspirin — TFN energy regression"); plt.tight_layout(); plt.show()

final_mae = hist[-1]
print(f"\nfinal test energy MAE: {final_mae:.4f} eV")
print("[Checkpoint D] target: energy MAE < 0.05 eV on aspirin (train 950 / test 50).")
if final_mae < 0.05:
    print("PASS — the equivariant network regresses a real DFT energy surface to chemical-ish accuracy.")
else:
    print("Above target — train more epochs, raise SH_LMAX to 3, or widen HIDDEN.")

### Baseline: why equivariance still earns its keep on a real PES

On Tetris the contrast was stark (12% vs 100%). On a *smooth* energy surface a distance-only
SchNet-type model already does fairly well, because a molecule's energy is dominated by bond
lengths and angles that distance features partly capture. The equivariant model's advantage
shows up as **data efficiency** and in the **forces**: because the network outputs the energy
as a genuine $\mathrm{O}(3)$ scalar, the forces obtained as $\vec F = -\nabla_{\vec r} E$ are
*automatically* correct `1o` vectors — they rotate with the molecule by construction, with no
augmentation. A non-equivariant model has no such guarantee and must *learn* it from data.

> **Optional baseline comparison.** As an exercise, strip the model to scalars only — set
> `HIDDEN = "32x0e"` and `SH_LMAX = 0` so the message keeps only the `0e` (distance-gated
> scalar) channel — retrain, and compare the energy MAE. This is the SchNet-type
> distance-only model on the *same* aspirin subset. You should find the equivariant model
> reaches a lower MAE for the same training budget; the gap is the data-efficiency claim,
> now on real molecules instead of toy shapes.

### E3 (TODO) — turn on **force** regression

We trained on energies only. But rMD17 also gives the **forces** $\vec F_i = -\partial
E/\partial \vec r_i$ in eV/Å, and forces are what actually drive a molecular-dynamics
simulation. Crucially, in our model the forces come **for free**: $E$ is a differentiable
function of the atomic positions, so `jax.grad` gives the analytic gradient, and the result
is guaranteed to be an equivariant `1o` vector field because $E$ is an `0e` scalar.

Your task: add a force term to the loss,
$$\mathcal{L} = \underbrace{(E_\theta - E)^2}_{\text{energy}} \;+\; \lambda\,
\frac{1}{N}\sum_{i=1}^{N}\big\lVert \underbrace{-\nabla_{\vec r_i} E_\theta}_{\hat{\vec F}_i}
- \vec F_i \big\rVert^2,$$
with e.g. $\lambda \approx 1$ (forces dominate in practice; tune it). Fill in the `# TODO`
lines below, retrain, and report the **force MAE in eV/Å**.

In [ ]:
# --- E3 (TODO): force-aware loss --------------------------------------------
# Forces are the negative gradient of the predicted energy w.r.t. atomic positions.
# jax.grad differentiates the SCALAR energy -> a (N, 3) force field, automatically 1o.

F_train_j = jnp.asarray(F_train)   # (N_TRAIN, N, 3) [eV/Å]
F_test_j  = jnp.asarray(F_test)

def predicted_forces(params, positions_3d):
    # F = -dE/dr.  grad of a scalar over the (N,3) position array.
    grad_E = jax.grad(lambda x: energy_of(params, x))(positions_3d)   # (N, 3)
    return -grad_E

def ef_loss(params, positions_3d, E_target, F_target, lam=1.0):
    E_pred = energy_of(params, positions_3d)
    energy_term = (E_pred - E_target) ** 2
    # TODO: compute the predicted forces with predicted_forces(...) and the
    # mean-squared-error force term  mean(||F_pred - F_target||^2).
    F_pred = ...          # TODO  (hint: predicted_forces(params, positions_3d))
    force_term = ...      # TODO  (hint: jnp.mean(jnp.sum((F_pred - F_target)**2, axis=-1)))
    return energy_term + lam * force_term

# Once the two TODO lines are filled, wrap ef_loss in a vmapped batch loss exactly
# like batch_loss above, swap it into train_step, retrain, then report:
#   force_mae = mean over test frames of mean_i ||F_pred_i - F_target_i||  (eV/Å)
# Expected: a few times 0.1 eV/Å for this tiny model/budget; foundation models
# (Lecture 13: MACE-MP-0) reach < 0.05 eV/Å on rMD17.
print("E3: fill in the two TODO lines, then retrain with the force-aware loss.")

### E4 — parity: chirality on Tetris vs aspirin

In §12.16 the single `0o` (parity-odd scalar) channel was what told `chiral_1` from its
mirror image `chiral_2`. Two experiments make the role of parity precise:

1. **On Tetris:** remove the `0o` channel (set the last layer to `"7x0e"` as in exercise E2)
   and retrain. The chiral pair collapses — the two mirror images get *identical* logits and
   the network can no longer reach 100%. Parity is **load-bearing** here.

2. **On rMD17 aspirin:** our energy readout is a pure `0e` scalar (`"1x0e"`), with **no**
   parity-odd channel at all — and yet the energy regression works fine. Why? Because the
   *energy* of a molecule is genuinely parity-**invariant**: $E(\text{molecule}) =
   E(\text{mirror image})$. A scalar PES never needs an odd channel.

The discussion point: **a physical symmetry is not always chirality.** Tetris was
*designed* to contain a chiral pair so that parity mattered; a generic energy surface is
parity-even and an `0e`-only readout is exactly right. Parity-odd channels become essential
only when you predict a **pseudoscalar** or **pseudovector** target — e.g. optical rotation,
a magnetic moment, or when distinguishing enantiomers by a chiroptical response. Knowing
*which* irrep your target lives in is the modelling decision; the network just enforces it.

## 12.B Recap — toy vs real, side by side

| | **Tetris** (toy/synthetic) | **rMD17 aspirin** (real data) |
|---|---|---|
| input | 4 atoms, integer grid | 21 atoms, Å coordinates from *ab initio* MD |
| task | 8-class shape classification | regress potential energy (+ optional forces) |
| target irrep | mixed (incl. `0o` for chirality) | `0e` scalar energy; `1o` forces via $-\nabla E$ |
| ground truth | hand-built labels | DFT (PBE/def2-SVP) |
| units | dimensionless | energy eV, force eV/Å |
| metric | accuracy | energy MAE [eV], force MAE [eV/Å] |
| what it proves | equivariance is **exact** (to $10^{-15}$) | equivariance is **useful** on real PES |
| typical result | 100% / 100% (train / rotated test) | energy MAE $\lesssim$ 0.05 eV (Checkpoint D) |

The same architecture spans both rows. That is the whole point of the symmetry blueprint
from §12.0: rotation + parity equivariance is a **structural prior** that pays off identically
on a toy shape set and on a real molecule.

> **Where this goes next (L13).** This network is trained *from scratch* on one molecule.
> Lecture 13 takes the same equivariant message-passing recipe to its production form —
> **NequIP / MACE** — and to the foundation model **MACE-MP-0**, which is pre-trained on
> millions of DFT structures across 89 elements. There we *fine-tune* MACE-MP-0 on a handful
> of new configurations instead of training from zero, and measure a real materials property
> (bulk modulus from an equation of state).

## 12.C Recap: the Tetris machinery

This chapter reproduced Thomas et al. (2018) *Tensor Field Networks* §4.1 in a 75-minute working session.

| Result | Why it works |
|---|---|
| 8 shapes classified from 8 training examples | every layer is rotation-equivariant by construction |
| Chiral pair distinguished | the `0o` parity-odd output channel flips sign under reflection |
| 100% accuracy on 100 random rotations | rotation does not change outputs by construction |
| Vanilla MLP collapses to about 12% on rotated test | no symmetry baked in |

## 12.19 Glossary of symbols

| Symbol | Meaning | First introduced |
|---|---|---|
| $R$ | $3\times 3$ rotation matrix, $R^\top R = I$, $\det R = +1$ | §12.9 |
| $\mathrm{SO}(3)$ | rotation group in 3D | §12.9 |
| $Y_\ell^m(\hat r)$ | real spherical harmonic | §12.10 |
| $D^\ell(R)$ | Wigner-D matrix; rotation acts on $Y_\ell$ as $D^\ell(R)$ | §12.11 |
| `0e`, `0o`, `1e`, `1o`, … | irrep labels (angular $\ell$ + parity) | §12.12 |
| $\otimes$ | equivariant tensor product, output decomposed by Clebsch–Gordan | §12.13 |
| $\oplus$ | direct sum / concatenation of irrep blocks | §12.13 |
| $h_i$ | atomic feature, an `IrrepsArray` | §12.15 |
| $\mathcal{N}(i)$ | neighbour set of atom $i$ within cutoff $r_c$ | §12.16 |
| $s_o,\,s_e,\,e_2,\ldots,e_7$ | readout components feeding the 8 class logits | §12.16 |

## 12.20 Why this scales

Modern molecular simulation uses this same recipe at full size.

* **NequIP** (Batzner et al., 2022). Equivariant message passing for atomic potentials. State-of-the-art accuracy with $10^2$–$10^3\times$ less training data than non-equivariant baselines on MD-17 and liquid water.
* **MACE** (Batatia et al., 2022). Same family. Carries up to 4-body correlations in one layer.
* **MACE-MP-0** (2023). A foundation model covering 89 elements; available on Hugging Face as `mace-mp-0`.

For molecules, materials and drug discovery the data-efficiency gap is decisive: training-set generation costs CPU-millennia of DFT.

* NequIP repository. <https://github.com/mir-group/nequip>
* MACE repository. <https://github.com/ACEsuit/mace>

## 12.21 Optional take-home exercises

**E1. Swap the chiral labels.** In the dataset cell, swap the labels of `chiral_1` and `chiral_2`, then retrain. Predict whether the network still hits 100%, and explain why in one sentence. *Hint:* what does the network learn the sign of $s_o$ to be?

**E2. Remove the `0o` channel.** Change the last layer's irreps from `"0o + 7x0e"` to `"7x0e"`. Retrain. Predict and verify what happens to the chiral pair. *Goal:* observe directly that `0o` is the carrier of chirality information.

**E3. Different $\ell_\mathrm{max}$.** Change `sh_lmax` in the layer from 3 to 1. What is the smallest value that still classifies all 8 shapes? Relate your answer to the angular content needed to break the chiral degeneracy.

**E4. Compare to L11's SchNet on the same axes.** Repeat the rotated-test plot of §12.17 with SchNet's predictions added as a third curve. The expected ordering is

$$\text{equivariant (100%)} \;>\; \text{SchNet (~87.5%)} \;>\; \text{vanilla MLP (~12.5%)},$$

and is a one-figure summary of the chapter.

> See also the **real-data checkpoints** added in §12.A: **Checkpoint D** (aspirin energy MAE < 0.05 eV), **E3** (force regression via $-\nabla E$), and **E4** (parity is load-bearing on Tetris but not on a scalar energy surface).

## 12.22 References

* **TFN** — Thomas, Smidt, Kearnes, Yang, Li, Kohlhoff, Riley. *Tensor Field Networks: Rotation- and Translation-Equivariant Neural Networks for 3D Point Clouds.* 2018. [arXiv:1802.08219](https://arxiv.org/abs/1802.08219)
* **SchNet** — Schütt, Sauceda, Kindermans, Tkatchenko, Müller. *SchNet: A continuous-filter convolutional neural network for modeling quantum interactions.* NeurIPS 2017. [arXiv:1706.08566](https://arxiv.org/abs/1706.08566)
* **DTNN** (distinct from SchNet) — Schütt, Arbabzadah, Chmiela, Müller, Tkatchenko. *Quantum-chemical insights from deep tensor neural networks.* Nat. Commun. 8, 13890 (2017). [doi:10.1038/ncomms13890](https://doi.org/10.1038/ncomms13890)
* **rMD17** — Christensen, von Lilienfeld. *On the role of gradients for machine learning of molecular energies and forces.* Mach. Learn.: Sci. Technol. 1, 045018 (2020). [doi:10.1088/2632-2153/abba6f](https://doi.org/10.1088/2632-2153/abba6f)
* **NequIP** — Batzner et al. *E(3)-equivariant graph neural networks for data-efficient and accurate interatomic potentials.* Nat. Commun. 13, 2453 (2022).
* **MACE** — Batatia et al. *MACE: Higher Order Equivariant Message Passing Neural Networks for Fast and Accurate Force Fields.* NeurIPS 2022.
* **e3nn** — Geiger, Smidt. *e3nn: Euclidean Neural Networks.* 2022. [arXiv:2207.09453](https://arxiv.org/abs/2207.09453)

![lineage from TFN to NequIP, MACE, and MACE-MP-0.](images/wrap_lineage.png)
